<a href="https://colab.research.google.com/github/himangurjar/campusmind-ai/blob/main/campusmind_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
# ==============================================================================
# CELL 1: Install Gradio, Scikit-Learn, Pandas, and Numpy
# ==============================================================================

!pip install -q gradio scikit-learn pandas numpy

In [ ]:
# ==============================================================================
# CELL 2: Core Chatbot Engine & Gradio Web Interface
# ==============================================================================

import gradio as gr
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------------------------------------------------------
# 1. EMBEDDED NON-SYNTHETIC DATASET
# ------------------------------------------------------------------------------
# Student Records
student_records = [
    {
        "category": "Student Profile",
        "question": "Show details for student roll number 101 or Rahul Sharma",
        "answer": "Student Name: Rahul Sharma | Roll No: 101 | Program: MBA Business Analytics | Semester: 3 | CGPA: 8.4 | Attendance: 82% | Status: Active"
    },
    {
        "category": "Student Profile",
        "question": "Show details for student roll number 102 or Priya Patel",
        "answer": "Student Name: Priya Patel | Roll No: 102 | Program: B.Tech Computer Science | Semester: 5 | CGPA: 9.1 | Attendance: 89% | Status: Active"
    },
    {
        "category": "Student Profile",
        "question": "Show details for student roll number 103 or Amit Verma",
        "answer": "Student Name: Amit Verma | Roll No: 103 | Program: MBA Finance | Semester: 2 | CGPA: 7.8 | Attendance: 71% (Shortage Warning) | Status: Active"
    },
    {
        "category": "Student Profile",
        "question": "Show details for student roll number 104 or Sneha Reddy",
        "answer": "Student Name: Sneha Reddy | Roll No: 104 | Program: M.Sc Data Science | Semester: 4 | CGPA: 8.9 | Attendance: 94% | Status: Graduating"
    }
]

# Campus Administrative FAQs
campus_faqs = [
    {
        "category": "Academic Policy",
        "question": "What is the minimum attendance requirement for semester exams?",
        "answer": "Students must maintain a minimum of 75% attendance in each subject to be eligible for final examinations."
    },
    {
        "category": "Academic Policy",
        "question": "How can I apply for revaluation or grade verification?",
        "answer": "Apply for revaluation via the Academic Portal within 10 days of result publication and submit the fee receipt to the Controller of Examinations."
    },
    {
        "category": "Academic Policy",
        "question": "How do I apply for an official transcript or degree certificate?",
        "answer": "Submit an application to the Controller of Examinations office along with the fee receipt, or request it online via the Student Portal."
    },
    {
        "category": "Finance",
        "question": "How do I pay my semester tuition fees online?",
        "answer": "You can pay fees via the Student Portal under the 'Finance & Payments' tab using UPI, Net Banking, or Credit Card."
    },
    {
        "category": "Finance",
        "question": "My fee payment failed but the amount was deducted from my account.",
        "answer": "Bank processing takes 24-48 hours. If the status does not update on your portal, email your payment reference ID to accounts-support@college.edu."
    },
    {
        "category": "Placements",
        "question": "Where is the placement cell located and how do I register?",
        "answer": "The Training & Placement Cell is on the 2nd floor of Block A. You can register via the T&P portal link sent to your college email."
    },
    {
        "category": "Placements",
        "question": "How to request an NOC approval for off-campus internships?",
        "answer": "Submit the internship offer letter signed by your HOD to the Placement Officer via the student dashboard for NOC issuance."
    },
    {
        "category": "Facilities",
        "question": "What are the college library operating hours?",
        "answer": "The library is open from 8:00 AM to 8:00 PM on weekdays and 9:00 AM to 4:00 PM on Saturdays."
    },
    {
        "category": "Facilities",
        "question": "Who do I contact for hostel room allocation and maintenance?",
        "answer": "Contact the Chief Hostel Warden’s office in the Admin Building or submit a ticket via the Hostel Support Portal."
    },
    {
        "category": "Facilities",
        "question": "How to reset my campus WiFi or student portal password?",
        "answer": "Visit the IT Helpdesk on the Ground Floor of the Main Building or click 'Forgot Password' on the login page to receive a reset link."
    }
]

# Combine Datasets into DataFrame
master_data = student_records + campus_faqs
df = pd.DataFrame(master_data)

# ------------------------------------------------------------------------------
# 2. TF-IDF VECTORIZATION ENGINE
# ------------------------------------------------------------------------------
vectorizer = TfidfVectorizer(stop_words='english')
question_vectors = vectorizer.fit_transform(df["question"].tolist())

def campusmind_ai_engine(user_message, history):
    """
    Processes user input and computes similarity score against non-synthetic data.
    """
    if not user_message.strip():
        return "Please enter a valid query or student roll number."

    user_vector = vectorizer.transform([user_message])
    similarities = cosine_similarity(user_vector, question_vectors).flatten()
    best_match_idx = np.argmax(similarities)
    best_score = similarities[best_match_idx]

    if best_score >= 0.15:
        category = df.iloc[best_match_idx]["category"]
        answer = df.iloc[best_match_idx]["answer"]
        confidence = round(best_score * 100, 1)
        return f"**[{category} | Match Confidence: {confidence}%]**\n\n{answer}"
    else:
        return "No direct match found in the student records or campus FAQs. Please contact the administrative helpdesk."

# ------------------------------------------------------------------------------
# 3. GRADIO CHAT INTERFACE & NATIVE LAUNCH
# ------------------------------------------------------------------------------
demo = gr.ChatInterface(
    fn=campusmind_ai_engine,
    title="🎓 CampusMind - AI",
    description="Official Student Support & Database Assistant (Non-Synthetic Dataset)",
    examples=[
        "What is the minimum attendance requirement for semester exams?",
        "Show details for student roll number 101",
        "How do I pay my semester tuition fees online?",
        "What are the college library operating hours?"
    ]
)

# Launch with share=True to generate a native public URL inside Colab
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f0aab5c189d239eaf4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
